In [2]:
import pandas as pd
import numpy as np

# ============================================================
# Configuration
#
# Script 1: builds county-level radiologist/pathologist provider
# counts from the NPPES workforce snapshot + HUD ZIP-to-County
# Crosswalk. This is the 2021-vintage version (no census zero-fill at
# this stage — that happens in Script 2). Three fixes are applied
# here (identical in structure to the 2026-vintage version of this
# script, differing only in input filenames):
#   1. ALLOC_RATIO: ZIP-group-level 3-tier fallback (bus->res->oth),
#      not the original row-level 2-tier np.where.
#   2. Per-subset aggregation: group by FIPS (COUNTY) alone, not by
#      the compound (COUNTY, USPS_ZIP_PREF_STATE) key.
#   3. Final combine step: merge the three provider-type tables on
#      FIPS alone; attach state once, at the end, from a standalone
#      FIPS->state lookup built from the full crosswalk.
# Fixes 2 and 3 both address the same root issue: USPS_ZIP_PREF_STATE
# reflects a ZIP's postal-preferred state (a mailing convention), not
# a county's true state. For boundary ZIPs that straddle a state line
# (e.g., Fort Campbell, ZIP 42223, USPS-labeled KY but sitting in
# Montgomery County, TN, FIPS 47125), treating state as part of a
# county's identity — either as a groupby key or a merge key — splits
# one true county into two rows, silently understating its true
# provider count and duplicating it in the final analytic sample.
# ============================================================
NPI_FILE       = 'radiologist_pathologist_filtered_2021_07.csv'   # 2021 NPPES snapshot, already taxonomy-filtered (script 0 output)
CROSSWALK_FILE = 'ZIP-COUNTY_062021.xlsx'                          # HUD ZIP-to-County Crosswalk, 2021 Q2 release
OUTPUT_FILE    = 'county_radiologist_pathologist_counts_2021_07.csv'
EXCLUDED_FILE  = 'county_counts_excluded_ambiguous_2021_07.csv'
UNALLOCATED_FILE = 'zips_with_zero_bus_res_ratio_2021_07.csv'

PRIMARY_CODE      = '2085R0202X'
PATHOLOGIST_CODES = {'207ZP0102X', '207ZP0101X', '207ZC0500X'}
ALL_TARGET_CODES  = {PRIMARY_CODE} | PATHOLOGIST_CODES

# CT intentionally included at this stage — excluded later in script 2
# via the census-side US_STATES set.
US_STATES = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','DC','FL','GA','HI','ID',
    'IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO',
    'MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA',
    'RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY'
}

TAXONOMY_COLS        = [f'Healthcare Provider Taxonomy Code_{i}' for i in range(1, 16)]
TAXONOMY_SWITCH_COLS = [f'Healthcare Provider Primary Taxonomy Switch_{i}' for i in range(1, 16)]

# ============================================================
# Step 1: Load data
# ============================================================
print("Loading data...")
npi   = pd.read_csv(NPI_FILE, low_memory=False, dtype=str)
xwalk = pd.read_excel(CROSSWALK_FILE, dtype=str)

# Note: the 2021 crosswalk uses lowercase zip/geoid/state/... column
# names — renamed here to match the standard ZIP/COUNTY/
# USPS_ZIP_PREF_STATE/RES_RATIO naming used throughout this script.
xwalk = xwalk.rename(columns={
    'zip': 'ZIP',
    'geoid': 'COUNTY',
    'state': 'USPS_ZIP_PREF_STATE',
    'res_ratio': 'RES_RATIO',
    'bus_ratio': 'BUS_RATIO',
    'oth_ratio': 'OTH_RATIO',
    'tot_ratio': 'TOT_RATIO',
})

for col in ['RES_RATIO', 'BUS_RATIO', 'OTH_RATIO', 'TOT_RATIO']:
    xwalk[col] = pd.to_numeric(xwalk[col], errors='coerce').fillna(0)

print(f"NPI records loaded: {len(npi):,}")
print(f"Crosswalk rows loaded: {len(xwalk):,} | Unique ZIPs: {xwalk['ZIP'].nunique():,} | Unique FIPS: {xwalk['COUNTY'].nunique():,}")

# ============================================================
# Step 2: Handle ambiguous providers
# Rule: providers with BOTH radiologist AND pathologist codes
#   → if primary taxonomy is one of the target codes: assign by primary
#   → if primary taxonomy is NOT in target codes: exclude from all groups
#
# Providers with only ONE group's code are always kept
# regardless of whether primary=Y or N
# ============================================================

has_rad  = npi[TAXONOMY_COLS].isin({PRIMARY_CODE}).any(axis=1)
has_path = npi[TAXONOMY_COLS].isin(PATHOLOGIST_CODES).any(axis=1)
has_both = has_rad & has_path

print(f"\nProviders with radiologist code only: {(has_rad & ~has_path).sum():,}")
print(f"Providers with pathologist code only: {(has_path & ~has_rad).sum():,}")
print(f"Providers with BOTH codes:            {has_both.sum():,}")

def get_primary_target(row):
    """Return the primary target code if it exists, else None"""
    for i, col in enumerate(TAXONOMY_COLS):
        if row[col] in ALL_TARGET_CODES:
            switch_col = TAXONOMY_SWITCH_COLS[i]
            if row[switch_col] == 'Y':
                return row[col]
    return None

ambiguous = npi[has_both].copy()
ambiguous['primary_target'] = ambiguous.apply(get_primary_target, axis=1)

resolvable = ambiguous[ambiguous['primary_target'].notna()].copy()
excluded   = ambiguous[ambiguous['primary_target'].isna()].copy()

print(f"\nOf the {has_both.sum()} providers with both codes:")
print(f"  Resolvable (primary found):     {len(resolvable):,}")
print(f"  Excluded (no primary in target): {len(excluded):,}")

excluded['exclusion_reason'] = 'ambiguous_dual_specialty_no_primary'
excluded.to_csv(EXCLUDED_FILE, index=False)
print(f"\nExcluded providers saved to: {EXCLUDED_FILE}")

# ============================================================
# Step 3: Build clean NPI subsets
# ============================================================

npi_single = npi[~has_both].copy()

resolvable['matched_taxonomy'] = resolvable['primary_target']
npi_clean = pd.concat([npi_single, resolvable], ignore_index=True)

print(f"\nClean NPI records for analysis: {len(npi_clean):,}")
print(f"(Original: {len(npi):,} | Excluded: {len(excluded):,})")

npi_rad         = npi_clean[npi_clean['matched_taxonomy'] == PRIMARY_CODE].copy()
npi_rad_primary = npi_rad[npi_rad['matched_taxonomy_is_primary'] == 'Y'].copy()
npi_path        = npi_clean[npi_clean['matched_taxonomy'].isin(PATHOLOGIST_CODES)].copy()

print(f"\nDiagnostic Radiologists - all:          {len(npi_rad):,}")
print(f"Diagnostic Radiologists - primary only: {len(npi_rad_primary):,}")
print(f"Pathologists - all:                     {len(npi_path):,}")

# ============================================================
# Step 4: Normalize allocation ratios
#
# FIX 1 (ALLOC_RATIO): the original notebooks used
# np.where(BUS_RATIO_NORM > 0, ...) — a row-level decision that could
# mix bus_ratio- and res_ratio-normalized rows within the same ZIP,
# causing ALLOC_RATIO to not sum to 1 across a ZIP's counties. Fixed
# to a ZIP-group-level decision (using the group's ratio sums, not
# each row's own normalized value), with a third fallback tier to
# oth_ratio for ZIPs where both bus_ratio and res_ratio sum to zero —
# the original notebooks had no such third tier at all.
# ============================================================
xwalk['BUS_RATIO_NORM'] = xwalk.groupby('ZIP')['BUS_RATIO'].transform(
    lambda x: x / x.sum() if x.sum() > 0 else x
)
xwalk['RES_RATIO_NORM'] = xwalk.groupby('ZIP')['RES_RATIO'].transform(
    lambda x: x / x.sum() if x.sum() > 0 else x
)
xwalk['OTH_RATIO_NORM'] = xwalk.groupby('ZIP')['OTH_RATIO'].transform(
    lambda x: x / x.sum() if x.sum() > 0 else x
)

zip_bus_sum = xwalk.groupby('ZIP')['BUS_RATIO'].transform('sum')
zip_res_sum = xwalk.groupby('ZIP')['RES_RATIO'].transform('sum')

xwalk['ALLOC_RATIO'] = np.select(
    [zip_bus_sum > 0, zip_res_sum > 0],
    [xwalk['BUS_RATIO_NORM'], xwalk['RES_RATIO_NORM']],
    default=xwalk['OTH_RATIO_NORM']
)

alloc_check = xwalk.groupby('ZIP')['ALLOC_RATIO'].sum()
bad_zips = alloc_check[(alloc_check - 1).abs() > 1e-6]
print(f"\n[Sanity check] ZIPs where ALLOC_RATIO does not sum to ~1 after fix: {len(bad_zips):,} (should be 0 or near-0)")

zero_bus_res = xwalk.groupby('ZIP').filter(lambda g: g['BUS_RATIO'].sum() == 0 and g['RES_RATIO'].sum() == 0)
zero_bus_res[['ZIP', 'COUNTY', 'USPS_ZIP_PREF_STATE']].drop_duplicates().to_csv(UNALLOCATED_FILE, index=False)
print(f"[Info] {zero_bus_res['ZIP'].nunique():,} ZIPs fell back to oth_ratio allocation (bus_ratio and res_ratio both 0). Saved to: {UNALLOCATED_FILE}")

# ============================================================
# Step 4b: Standalone FIPS -> state lookup, built from the full
# crosswalk (covers every FIPS in the crosswalk, regardless of
# whether it has any matching providers). Used once, at the very end
# (Step 7), instead of carrying a per-subset state label through the
# aggregation — see FIX 2 and FIX 3 below for why.
# ============================================================
county_state_lookup = xwalk.groupby('COUNTY')['USPS_ZIP_PREF_STATE'].first().reset_index()
county_state_lookup.columns = ['FIPS', 'state']

# ============================================================
# Step 5: Merge and aggregate function
#
# FIX 2: group by FIPS (COUNTY) alone, not by (COUNTY,
# USPS_ZIP_PREF_STATE) together. State is not carried through here at
# all — it is attached once at the end (Step 7) from the standalone
# lookup above, so a boundary county's providers always sum into a
# single row regardless of which ZIP-level postal-state label any
# individual provider's ZIP happens to carry.
# ============================================================
def merge_and_aggregate(npi_df, label):
    merged = npi_df.merge(
        xwalk[['ZIP', 'COUNTY', 'ALLOC_RATIO']],
        left_on='zip5',
        right_on='ZIP',
        how='left'
    )

    unmatched = merged['COUNTY'].isna().sum()
    print(f"\n[{label}] Unmatched ZIPs: {unmatched:,} ({unmatched/len(npi_df)*100:.1f}%)")

    merged = merged.dropna(subset=['COUNTY'])
    merged['provider_count'] = merged['ALLOC_RATIO']

    county_counts = merged.groupby('COUNTY')['provider_count'].sum().reset_index()
    county_counts.columns = ['FIPS', f'count_{label}']
    county_counts[f'count_{label}'] = county_counts[f'count_{label}'].round(4)

    print(f"[{label}] Counties with >=1 provider: {len(county_counts):,}")
    print(f"[{label}] Total allocated count:       {county_counts[f'count_{label}'].sum():.1f}")

    return county_counts

# ============================================================
# Step 6: Run aggregation
# ============================================================
counts_rad         = merge_and_aggregate(npi_rad,         'radiologist')
counts_rad_primary = merge_and_aggregate(npi_rad_primary, 'radiologist_primary')
counts_path        = merge_and_aggregate(npi_path,        'pathologist')

# ============================================================
# Step 7: Combine into single county file
#
# FIX 3: merge the three count tables on FIPS alone — none of them
# carry a 'state' column anymore, so there is nothing for a compound
# key to mismatch on. State is attached once, after combining, from
# the standalone lookup built in Step 4b (covers every FIPS in the
# crosswalk, so a county with e.g. only pathologist providers still
# gets a valid state and is not dropped by the US_STATES filter below).
# ============================================================
county_df = counts_rad \
    .merge(counts_rad_primary, on='FIPS', how='outer') \
    .merge(counts_path,        on='FIPS', how='outer')

county_df = county_df.fillna(0)
county_df = county_df.merge(county_state_lookup, on='FIPS', how='left')

missing_state = county_df['state'].isna().sum()
if missing_state > 0:
    print(f"\n[Warning] {missing_state} counties with providers have no state match in the crosswalk lookup — investigate before proceeding.")

county_df = county_df[county_df['state'].isin(US_STATES)]

print(f"\nCounties with at least one provider: {len(county_df):,}")
print(f"Duplicated FIPS in final table: {county_df['FIPS'].duplicated().sum():,} (should be 0)")

# Note: no census zero-fill step here — county universe construction
# and CT exclusion both happen in script 2, via a left join from the
# census side.

# ============================================================
# Step 8: Summary
# ============================================================
print("\n=== Radiologist Count Distribution by County ===")
print(county_df['count_radiologist'].describe().round(2))

print(f"\nCounties with zero radiologists (main):         {(county_df['count_radiologist'] == 0).sum():,}")
print(f"Counties with zero radiologists (primary only): {(county_df['count_radiologist_primary'] == 0).sum():,}")
print(f"Counties with zero pathologists:                {(county_df['count_pathologist'] == 0).sum():,}")

# ============================================================
# Step 9: Save
# ============================================================
county_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved to: {OUTPUT_FILE}")
print(f"Columns: {county_df.columns.tolist()}")
print("\nNext step: run Script 2 (census_provider_population_ratio) to merge with Census population data")

Loading data...
NPI records loaded: 61,721
Crosswalk rows loaded: 54,212 | Unique ZIPs: 39,448 | Unique FIPS: 3,228

Providers with radiologist code only: 42,843
Providers with pathologist code only: 18,871
Providers with BOTH codes:            7

Of the 7 providers with both codes:
  Resolvable (primary found):     4
  Excluded (no primary in target): 3

Excluded providers saved to: county_counts_excluded_ambiguous_2021_07.csv

Clean NPI records for analysis: 61,718
(Original: 61,721 | Excluded: 3)

Diagnostic Radiologists - all:          42,845
Diagnostic Radiologists - primary only: 39,311
Pathologists - all:                     18,873

[Sanity check] ZIPs where ALLOC_RATIO does not sum to ~1 after fix: 0 (should be 0 or near-0)
[Info] 1,885 ZIPs fell back to oth_ratio allocation (bus_ratio and res_ratio both 0). Saved to: zips_with_zero_bus_res_ratio_2021_07.csv

[radiologist] Unmatched ZIPs: 74 (0.2%)
[radiologist] Counties with >=1 provider: 1,914
[radiologist] Total allocated co